# Data Mining Assignment: Apache Spark Data Lake Preprocessing

**Course:** B.Tech 7th Semester Data Mining Lab  
**Student Name:** Sankaramaddi Yamini  
**Roll Number:** LCS2023031  
**Assignment:** Apache Spark Data Lake Preprocessing in Google Colab  
**Dataset:** `sales.csv`  
**Environment:** Google Colab + PySpark  

---

## Architecture Overview: Medallion Data Lake
This notebook implements a two-tier Data Lake architecture:
1. **Bronze Layer (`data_lake/bronze/`):** Raw, immutable data ingestion layer where `sales.csv` is preserved exactly as provided.
2. **Silver Layer (`data_lake/silver/`):** Cleaned, validated, and standardized data preprocessed using Apache Spark.

```
data_lake/
├── bronze/
│   └── sales.csv             <- Raw, unchanged dataset
└── silver/
    └── preprocessed_sales/   <- Cleaned Silver dataset (CSV/Parquet)
```

In [1]:
# =============================================================================
# STUDENT SUBMISSION CONFIGURATION
# =============================================================================
ROLL_NUMBER = "LCS2023031"
STUDENT_NAME = "Sankaramaddi_Yamini"

print(f"Submission target: submissions/{ROLL_NUMBER}_{STUDENT_NAME}/")

Submission target: submissions/LCS2023031_Sankaramaddi_Yamini/


## 1. Environment Setup & Data Lake Structure
- Install PySpark in the runtime environment
- Create the Data Lake folder structure: `data_lake/bronze/` and `data_lake/silver/preprocessed_sales/`
- Ingest the raw `sales.csv` into the Bronze layer without modification

In [2]:
# Install PySpark
!pip install -q pyspark

import os
import shutil
import glob
import urllib.request
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Create Data Lake directories
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Locate or download raw sales.csv and place into Bronze layer
candidate_paths = [
    "dataset/sales.csv",
    "data-mining-data-lake-assignment/dataset/sales.csv",
    "/content/dataset/sales.csv",
    "/content/sales.csv",
    "sales.csv"
]

source_dataset = next((p for p in candidate_paths if os.path.exists(p)), None)
bronze_path = "data_lake/bronze/sales.csv"

if source_dataset:
    shutil.copy(source_dataset, bronze_path)
    print(f"Successfully ingested raw dataset into Bronze Layer: {bronze_path}")
else:
    raw_url = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
    urllib.request.urlretrieve(raw_url, bronze_path)
    print(f"Downloaded and ingested raw dataset into Bronze Layer from GitHub: {bronze_path}")

print("\nData Lake Directory Structure Created:")
print("data_lake/")
print("├── bronze/")
print("│   └── sales.csv (Raw, Unchanged)")
print("└── silver/")
print("    └── preprocessed_sales/")

Successfully ingested raw dataset into Bronze Layer: data_lake/bronze/sales.csv

Data Lake Directory Structure Created:
data_lake/
├── bronze/
│   └── sales.csv (Raw, Unchanged)
└── silver/
    └── preprocessed_sales/


## 2. Load and Inspect with Spark
- Initialize SparkSession with legacy time parser support
- Read `sales.csv` from Bronze layer
- Display schema, first few rows, and total record count
- Identify discrepancies using Spark operations (no hard-coding)

In [3]:
# Initialize SparkSession
spark = SparkSession.builder \
    .appName("Apache_Spark_Data_Lake_Preprocessing") \
    .config("spark.sql.ansi.enabled", "false") \
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED") \
    .getOrCreate()

# Set log level to minimize verbosity
spark.sparkContext.setLogLevel("WARN")
print("SparkSession successfully created!")

# Read sales.csv from Bronze layer
bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Total record count
bronze_count = bronze_df.count()
print(f"Total Bronze Record Count: {bronze_count}")

# Display schema
print("\n--- Bronze Layer Schema ---")
bronze_df.printSchema()

# Display sample rows
print("\n--- Bronze Layer Sample (First 5 Rows) ---")
bronze_df.show(5, truncate=False)

SparkSession successfully created!
Total Bronze Record Count: 1000

--- Bronze Layer Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

--- Bronze Layer Sample (First 5 Rows) ---
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+-------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state        |order_date

### 2.1 Identifying Data Discrepancies via Spark Operations

We systematically identify the 5 types of anomalies using PySpark functions:
1. **Exact Duplicates:** Identified via `count() - dropDuplicates().count()`
2. **Null / Missing Values:** Identified via per-column null evaluation
3. **Text Formatting Inconsistencies:** Identified via whitespace comparisons and distinct value exploration
4. **Invalid Numerical Values:** Identified via boundary condition filters (`quantity <= 0`, `unit_price <= 0`, `discount_percent < 0 or > 100`)
5. **Inconsistent and Invalid Dates:** Identified via regex pattern matching and multi-format parsing

In [4]:
# 1. Identify Duplicate Records
total_rows = bronze_df.count()
distinct_rows = bronze_df.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")
print(f"Exact Duplicate Rows Identified: {duplicate_count}")

# Display sample duplicate rows by grouping
print("\nSample Duplicate Rows (grouped by all columns with count > 1):")
bronze_df.groupBy(bronze_df.columns).count().filter(F.col("count") > 1).show(5, truncate=False)

Total Rows: 1000
Distinct Rows: 990
Exact Duplicate Rows Identified: 10

Sample Duplicate Rows (grouped by all columns with count > 1):
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+---------+-------------+----------+------------+-----+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city     |state        |order_date|order_status|count|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+---------+-------------+----------+------------+-----+
|100055  |C0270      |Pranav Iyer   |T-Shirt      |Clothing|8       |1480      |15              |UPI           |Bengaluru|Karnataka    |2026-08-20|Delivered   |2    |
|100297  |C0149      |Keerthi Sharma|Sports Bottle|Sports  |5       |570       |0               |Net Banking   |Hyderabad|Telangana    |2025-07-29|Delivered   |2    |
|100588  |C0228      |Diya Tr

In [5]:
# 2. Identify Null / Missing Values per Column
null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in bronze_df.columns
])

print("Null / Empty String Counts across Columns:")
null_counts.show()

Null / Empty String Counts across Columns:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+


In [6]:
# 3. Identify Text-Format Inconsistencies (Whitespace & Capitalization)
string_columns = [f.name for f in bronze_df.schema.fields if isinstance(f.dataType, StringType)]

print("--- Leading/Trailing Whitespace Checks ---")
for c in string_columns:
    ws_count = bronze_df.filter(F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).count()
    if ws_count > 0:
        print(f"Column '{c}' contains {ws_count} records with leading/trailing spaces.")

print("\n--- Capitalization Inconsistencies in 'category' ---")
bronze_df.groupBy("category").count().show(truncate=False)

print("--- Capitalization and Whitespace Inconsistencies in 'city' ---")
bronze_df.groupBy("city").count().filter(F.col("city").isNotNull()).show(15, truncate=False)

--- Leading/Trailing Whitespace Checks ---
Column 'customer_name' contains 3 records with leading/trailing spaces.
Column 'city' contains 3 records with leading/trailing spaces.

--- Capitalization Inconsistencies in 'category' ---
+---------------+-----+
|category       |count|
+---------------+-----+
|sports         |1    |
|ELECTRONICS    |1    |
|CLOTHING       |1    |
|Home Appliances|115  |
|Beauty         |136  |
|Grocery        |136  |
|Books          |128  |
|Furniture      |132  |
|SPORTS         |1    |
|beauty         |2    |
|Sports         |136  |
|Clothing       |108  |
|Electronics    |103  |
+---------------+-----+

--- Capitalization and Whitespace Inconsistencies in 'city' ---
+------------+-----+
|city        |count|
+------------+-----+
|DELHI       |2    |
| Hyderabad  |2    |
| Mumbai     |1    |
|HYDERABAD   |1    |
|coimbatore  |1    |
|delhi       |1    |
|JAIPUR      |1    |
|Chennai     |88   |
|Vijayawada  |105  |
|Bengaluru   |91   |
|Lucknow     |90   |
|

In [7]:
# 4. Identify Invalid Numerical Values
# Quantity <= 0
invalid_qty = bronze_df.filter((F.col("quantity") <= 0) | F.col("quantity").isNull())
print(f"Invalid Quantity Records (quantity <= 0): {invalid_qty.count()}")
invalid_qty.select("order_id", "product", "quantity", "unit_price").show(5)

# Unit Price <= 0
invalid_price = bronze_df.filter((F.col("unit_price") <= 0) | F.col("unit_price").isNull())
print(f"Invalid Unit Price Records (unit_price <= 0): {invalid_price.count()}")
invalid_price.select("order_id", "product", "quantity", "unit_price").show(5)

# Discount Percent < 0 or > 100
invalid_discount = bronze_df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100))
print(f"Invalid Discount Percent Records (< 0 or > 100): {invalid_discount.count()}")
invalid_discount.select("order_id", "product", "discount_percent").show(10)

Invalid Quantity Records (quantity <= 0): 12
+--------+---------------+--------+----------+
|order_id|product        |quantity|unit_price|
+--------+---------------+--------+----------+
|100114  |Smart Watch    |0       |4240      |
|100143  |Air Conditioner|0       |48210     |
|100198  |Running Shoes  |-2      |2450      |
|100365  |Coffee 500g    |0       |950       |
|100375  |Wireless Mouse |0       |1390      |
+--------+---------------+--------+----------+
only showing top 5 rows

Invalid Unit Price Records (unit_price <= 0): 8
+--------+----------------------+--------+----------+
|order_id|product               |quantity|unit_price|
+--------+----------------------+--------+----------+
|100102  |Atta 10kg             |2       |0         |
|100170  |Competitive Exam Guide|4       |0         |
|100235  |Power Bank            |4       |-500      |
|100284  |Study Table           |9       |-500      |
|100487  |Office Chair          |7       |-500      |
+--------+-----------------

In [8]:
# 5. Identify Inconsistent / Invalid Date Values
# Identify records that do not adhere to standard YYYY-MM-DD pattern
non_std_date_pattern = bronze_df.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
print(f"Non-standard Date Strings Count: {non_std_date_pattern.count()}")
non_std_date_pattern.select("order_id", "customer_name", "order_date").show(truncate=False)

# Multi-format candidate parsing to identify parseable vs impossible dates
multi_format_parsed = F.coalesce(
    F.to_date(F.col("order_date"), "yyyy-MM-dd"),
    F.to_date(F.col("order_date"), "yyyy/MM/dd"),
    F.to_date(F.col("order_date"), "MM-dd-yyyy"),
    F.to_date(F.col("order_date"), "dd/MM/yyyy")
)

impossible_dates = bronze_df.filter(multi_format_parsed.isNull())
print(f"Clearly Invalid / Impossible Dates Count: {impossible_dates.count()}")
impossible_dates.select("order_id", "customer_name", "order_date").show(truncate=False)

Non-standard Date Strings Count: 6
+--------+---------------+----------+
|order_id|customer_name  |order_date|
+--------+---------------+----------+
|100097  |Shreya Rao     |31/02/2026|
|100358  |Sanjay Kulkarni|2025/07/14|
|100448  |Pranav Iyer    |not-a-date|
|100634  |Tanvi Mehta    |2025-02-30|
|100637  |Aarav Patel    |2026-13-05|
|100969  |Deepak Rao     |09-15-2026|
+--------+---------------+----------+

Clearly Invalid / Impossible Dates Count: 4
+--------+-------------+----------+
|order_id|customer_name|order_date|
+--------+-------------+----------+
|100097  |Shreya Rao   |31/02/2026|
|100448  |Pranav Iyer  |not-a-date|
|100634  |Tanvi Mehta  |2025-02-30|
|100637  |Aarav Patel  |2026-13-05|
+--------+-------------+----------+


## 3. Basic Preprocessing - Silver Layer

We systematically apply the required preprocessing rules:
1. **Remove exact duplicate rows:** via `dropDuplicates()`.
2. **Handle missing values:** Impute missing text fields (`customer_name`, `payment_method`, `city`) with `'Unknown'` to preserve valid transaction revenue without data loss.
3. **Trim extra spaces and standardize capitalization:**
   - Apply `trim()` to all string columns.
   - Apply `initcap()` to `category`, `city`, `state`, and `order_status`.
   - Preserve `UPI` standard acronym in `payment_method`.
4. **Validate numerical values:** Retain only physically valid business records (`quantity > 0`, `unit_price > 0`, and `0 <= discount_percent <= 100`).
5. **Standardize and validate dates:** Convert multi-format dates (`yyyy-MM-dd`, `yyyy/MM/dd`, `MM-dd-yyyy`) to `yyyy-MM-dd` and filter out impossible calendar dates (`31/02/2026`, `not-a-date`, `2025-02-30`, `2026-13-05`).
6. **Preserve useful records:** Keep valid business transactions intact without over-cleansing.

In [9]:
# -----------------------------------------------------------------------------
# Step 1: Remove exact duplicate rows
# -----------------------------------------------------------------------------
silver_s1 = bronze_df.dropDuplicates()
count_s1 = silver_s1.count()
print(f"[Step 1] Exact duplicates removed: {bronze_count - count_s1} rows. Remaining: {count_s1}")

# -----------------------------------------------------------------------------
# Step 2: Handle missing values (preserve records by imputing 'Unknown')
# -----------------------------------------------------------------------------
silver_s2 = silver_s1.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})
print("[Step 2] Missing values handled in customer_name, payment_method, and city.")

# -----------------------------------------------------------------------------
# Step 3: Trim extra spaces and standardize capitalization in text fields
# -----------------------------------------------------------------------------
silver_s3 = silver_s2
for col_name in string_columns:
    silver_s3 = silver_s3.withColumn(col_name, F.trim(F.col(col_name)))

# Standardize capitalization (initcap) and preserve standard UPI acronym
silver_s3 = silver_s3.withColumn("category", F.initcap(F.col("category"))) \
                     .withColumn("city", F.initcap(F.col("city"))) \
                     .withColumn("state", F.initcap(F.col("state"))) \
                     .withColumn("order_status", F.initcap(F.col("order_status"))) \
                     .withColumn("payment_method", \
                         F.when(F.upper(F.col("payment_method")) == "UPI", "UPI") \
                          .otherwise(F.initcap(F.col("payment_method"))))
print("[Step 3] Text fields trimmed and capitalization standardized.")

# -----------------------------------------------------------------------------
# Step 4: Apply basic validation to quantity, unit price, and discount percentage
# -----------------------------------------------------------------------------
silver_s4 = silver_s3.filter(
    (F.col("quantity") > 0) &
    (F.col("unit_price") > 0) &
    (F.col("discount_percent") >= 0) &
    (F.col("discount_percent") <= 100)
)
count_s4 = silver_s4.count()
print(f"[Step 4] Numerical validation applied. Removed: {count_s1 - count_s4} invalid rows. Remaining: {count_s4}")

# -----------------------------------------------------------------------------
# Step 5: Convert valid dates to consistent format and filter invalid dates
# -----------------------------------------------------------------------------
parsed_date_expr = F.coalesce(
    F.to_date(F.col("order_date"), "yyyy-MM-dd"),
    F.to_date(F.col("order_date"), "yyyy/MM/dd"),
    F.to_date(F.col("order_date"), "MM-dd-yyyy"),
    F.to_date(F.col("order_date"), "dd/MM/yyyy")
)
silver_df = silver_s4.withColumn("order_date_parsed", parsed_date_expr) \
                     .filter(F.col("order_date_parsed").isNotNull()) \
                     .withColumn("order_date", F.date_format(F.col("order_date_parsed"), "yyyy-MM-dd")) \
                     .drop("order_date_parsed") \
                     .sort("order_id")

final_silver_count = silver_df.count()
print(f"[Step 5] Date validation applied. Removed: {count_s4 - final_silver_count} invalid dates. Final Silver Count: {final_silver_count}")

[Step 1] Exact duplicates removed: 10 rows. Remaining: 990
[Step 2] Missing values handled in customer_name, payment_method, and city.
[Step 3] Text fields trimmed and capitalization standardized.
[Step 4] Numerical validation applied. Removed: 28 invalid rows. Remaining: 962
[Step 5] Date validation applied. Removed: 4 invalid dates. Final Silver Count: 958


### 3.2 Saving to the Silver Layer
- Save the preprocessed DataFrame into `data_lake/silver/preprocessed_sales`
- Export clean single CSV `silver/sales_silver.csv` into `submissions/<ROLL_NUMBER>_<NAME>/` for submission

In [10]:
# Save DataFrame in Silver layer folder in CSV format
silver_output_dir = "data_lake/silver/preprocessed_sales"

# Prepare submission directory: submissions/<ROLL_NUMBER>_<NAME>/silver/
sub_dir = f"submissions/{ROLL_NUMBER}_{STUDENT_NAME}"
sub_silver_dir = os.path.join(sub_dir, "silver")
os.makedirs(sub_silver_dir, exist_ok=True)
submission_csv = os.path.join(sub_silver_dir, "sales_silver.csv")

# In Linux / Google Colab environments, Spark write produces part files
try:
    silver_df.coalesce(1).write.csv(silver_output_dir, header=True, mode="overwrite")
    print(f"Silver DataFrame saved to {silver_output_dir} in CSV format.")
    part_files = glob.glob(f"{silver_output_dir}/part-*.csv")
    if part_files:
        shutil.copy(part_files[0], submission_csv)
        print(f"Submission Silver CSV created at: {submission_csv}")
except Exception as e:
    # Fallback writing via pandas if Spark native file commit is restricted
    silver_pdf = silver_df.toPandas()
    silver_pdf.to_csv(os.path.join(silver_output_dir, "sales_silver.csv"), index=False)
    silver_pdf.to_csv(submission_csv, index=False)
    print(f"Submission Silver CSV created at: {submission_csv}")

Silver DataFrame saved to data_lake/silver/preprocessed_sales in CSV format.
Submission Silver CSV created at: submissions/LCS2023031_Sankaramaddi_Yamini/silver/sales_silver.csv


## 4. Verification
- Compare row counts before and after preprocessing
- Read Silver data back with Spark and display schema, null counts, and sample rows
- Verify 0 remaining nulls and standardized formats

In [11]:
# 1. Show row count before and after preprocessing
print("=" * 65)
print(f"Record Count Before Preprocessing (Bronze): {bronze_count}")
print(f"Record Count After Preprocessing (Silver):  {final_silver_count}")
print(f"Total Invalid / Duplicate Records Removed: {bronze_count - final_silver_count}")
print("=" * 65)

# 2. Read the Silver data back with Spark and display schema and sample rows
silver_verified_df = spark.read.csv(submission_csv, header=True, inferSchema=True)
print("\n--- Verified Silver Layer Schema ---")
silver_verified_df.printSchema()

print("\n--- Verified Silver Layer Sample (First 10 Rows) ---")
silver_verified_df.show(10, truncate=False)

# Check nulls in Silver layer
print("\n--- Silver Layer Null Counts ---")
silver_verified_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_verified_df.columns
]).show()

Record Count Before Preprocessing (Bronze): 1000
Record Count After Preprocessing (Silver):  958
Total Invalid / Duplicate Records Removed: 42

--- Verified Silver Layer Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

--- Verified Silver Layer Sample (First 10 Rows) ---
+--------+-----------+---------------+--------------------+--------+--------+----------+----------------+----------------+----------+-------------+----------+------------+
|order_id|customer_id|customer_name  |prod

## 5. Summary of Preprocessing Operations Performed

| Category | Discrepancy Found in Bronze | Preprocessing Operation in Silver | Impact / Result |
| :--- | :--- | :--- | :--- |
| **Duplicates** | 10 exact duplicate rows | `dropDuplicates()` | Removed 10 redundant rows (1000 -> 990) |
| **Missing Values** | Nulls in `customer_name` (10), `payment_method` (8), `city` (8) | `fillna({'customer_name': 'Unknown', ...})` | Preserved all 26 transaction records while eliminating null values |
| **Whitespace** | Extra leading/trailing spaces in `city`, `customer_name` | `F.trim()` applied to all string columns | Cleaned leading and trailing whitespace |
| **Capitalization** | Inconsistent casing in `category` ('sports', 'ELECTRONICS') and `city` ('DELHI', 'HYDERABAD') | `F.initcap()` applied to category, city, state, order_status | Standardized to Title Case; preserved standard `UPI` acronym |
| **Numerical Integrity** | Non-positive `quantity` (<=0), non-positive `unit_price` (<=0), and invalid `discount_percent` (<0 or >100) | Filtered: `quantity > 0`, `unit_price > 0`, `0 <= discount_percent <= 100` | Removed 28 invalid rows (12 qty, 8 price, 8 discount) |
| **Date Integrity** | Alternate formats (`YYYY/MM/DD`, `MM-DD-YYYY`) and impossible dates (`31/02/2026`, `not-a-date`, etc.) | `coalesce` with multiple date patterns; filtered unparseable dates; formatted as `yyyy-MM-dd` | Standardized 2 valid non-standard dates; removed 4 impossible dates |
| **Record Preservation** | Legitimate business variations (cancelled orders, bulk purchases) | Preserved all realistic transactions | High data fidelity without over-filtering |